# TCG Grader · Colab Free V442\n태블릿 TCG Grader의 **선택형 무료 보조 학습/검증 노드**입니다.\n- 태블릿 실시간 런타임을 대체하지 않습니다.\n- Google Drive는 입력/체크포인트/후보 결과 저장에만 사용합니다.\n- 결과는 후보(candidate)로만 저장되며 태블릿이나 main에 자동 적용하지 않습니다.\n- Colab 무료 GPU/세션 가용성은 Google 정책과 계정 사용량에 따라 달라질 수 있습니다.\n

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive', force_remount=False)\nfrom pathlib import Path\nDRIVE_ROOT = Path('/content/drive/MyDrive/TCG_Grader_Colab')\nfor name in ('input','candidates','receipts','checkpoints'): (DRIVE_ROOT/name).mkdir(parents=True, exist_ok=True)\nprint('Drive root:', DRIVE_ROOT)\n

In [ ]:
import os, shutil, subprocess, sys, time, json, hashlib\nREPO_URL='https://github.com/wlghks24/-tcg-grader.git'\nREF='selfrefine/v429-registry-driven-market-lens'  # V442 검증 후 main 병합 시 main으로 변경 가능\nREPO=Path('/content/tcg-grader')\nif REPO.exists(): shutil.rmtree(REPO)\nsubprocess.run(['git','clone','--depth','1','--branch',REF,REPO_URL,str(REPO)],check=True)\norigin=subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()\nassert origin in {REPO_URL,'https://github.com/wlghks24/-tcg-grader'}, origin\nHEAD=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()\nsys.path.insert(0,str(REPO))\nimport colab_free_guard_v442 as guard\nPOLICY=guard.load_policy(REPO/'COLAB_FREE_POLICY_V442.json')\nSTART=time.monotonic()\nprint('repo head:',HEAD)\nprint('budget:',guard.session_budget(START,POLICY))\n

In [ ]:
def gpu_available():\n    if not shutil.which('nvidia-smi'): return False\n    return subprocess.run(['nvidia-smi','-L'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0\nGPU=gpu_available()\nMODE='GPU_AVAILABLE' if GPU else 'CPU_FALLBACK'\nprint('mode:',MODE)\nprint('Colab Free availability is dynamic; this notebook never requests paid accelerators.')\n

In [ ]:
def run_bounded(args, est_minutes=5, input_bytes=0):\n    ok,reason=guard.can_start_job(START,est_minutes,input_bytes,POLICY)\n    if not ok: return {'ok':False,'reason':reason,'cmd':args}\n    remaining=guard.session_budget(START,POLICY)['remaining_work_minutes']\n    timeout=max(30,int(min(POLICY['limits']['max_single_job_minutes'],remaining)*60))\n    cp=subprocess.run(args,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)\n    print(cp.stdout[-5000:])\n    return {'ok':cp.returncode==0,'returncode':cp.returncode,'cmd':args}\nchecks=[]\nchecks.append(run_bounded([sys.executable,'tablet_runtime_manifest.py','--check','--compile'],4))\ntests=[p for p in ('test_verified_grade_learning_v135_safe.py','test_grading_hierarchy_v17.py','test_tcg_market_council_v436.py','test_tcg_discovery_inbox_v437.py','test_colab_free_guard_v442.py') if (REPO/p).is_file()]\nif tests: checks.append(run_bounded([sys.executable,'-m','unittest','-v',*tests],10))\nassert all(x.get('ok') for x in checks), checks\n

In [ ]:
# Drive input이 있을 때만 verified-only calibration 후보를 생성합니다.\ninp=DRIVE_ROOT/'input'\nrequired=['learning_store.json','verified_certifications.json']\npresent=all((inp/n).is_file() for n in required)\ncandidate_written=False\ncandidate_dir=None\nif present:\n    total=sum((inp/n).stat().st_size for n in required)\n    ok,reason=guard.can_start_job(START,20,total,POLICY)\n    if not ok: print('calibration skipped:',reason)\n    else:\n        for n in required: shutil.copy2(inp/n,REPO/n)\n        import importlib, verified_grade_learning_v135_safe as safe\n        importlib.reload(safe)\n        audit=safe.audit()\n        print('verified training rows:',audit.get('verified_training_rows'))\n        if int(audit.get('verified_training_rows') or 0)>0:\n            result=safe.rebuild_safe_vision_calibration()\n            stamp=time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())\n            candidate_dir=DRIVE_ROOT/'candidates'/f'{stamp}_{HEAD[:12]}'\n            candidate_dir.mkdir(parents=True,exist_ok=False)\n            out=REPO/'vision_calibration.json'\n            if out.is_file() and out.stat().st_size<=POLICY['limits']['max_output_bytes']:\n                shutil.copy2(out,candidate_dir/'vision_calibration.json')\n                candidate_written=True\nelse:\n    print('No Drive calibration input; verification-only run completed.')\n

In [ ]:
receipt={\n 'schema_version':1,'policy_version':'v442','repo_head':HEAD,'ref':REF,'mode':MODE,\n 'candidate_written':candidate_written,'candidate_dir':str(candidate_dir) if candidate_dir else None,\n 'checks':checks,'budget':guard.session_budget(START,POLICY),\n 'paid_features_used':False,'github_write':False,'tablet_auto_apply':False,\n 'completed_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}\nraw=json.dumps(receipt,ensure_ascii=False,indent=2,allow_nan=False)+'\\n'\nname=f"COLAB_V442_RECEIPT_{time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())}_{HEAD[:12]}.json"\n(DRIVE_ROOT/'receipts'/name).write_text(raw,encoding='utf-8')\nprint(raw)\nprint('DONE: candidate only; nothing was pushed to GitHub or applied to the tablet.')\n